In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Task6_Tableau")
    .config("spark.sql.shuffle.partitions","8")
    .config("spark.driver.memory","4g")
    .config("spark.executor.memory","4g")
    .getOrCreate()
)

print("Spark Started Successfully")

Spark Started Successfully


In [4]:
import pandas as pd
import os

from pyspark.sql.functions import *

In [5]:
processed_df = spark.read.parquet(
    "/content/drive/MyDrive/7006SCN/processed_reviews"
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [7]:
processed_df.printSchema()

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = true)
 |-- label: double (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)



In [6]:
processed_df.show(5)

+------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+--------------------+--------------------+
|rating|                text|               title|              review|label|               words|      filtered_words|         rawFeatures|            features|
+------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+--------------------+--------------------+
|   3.0|Depending on your...|it tastes exactly...|it tastes exactly...|  2.0|[it, tastes, exac...|[tastes, exactly,...|(10000,[36,274,28...|(10000,[36,274,28...|
|   5.0|Excellent value f...|   Excellent product|excellent product...|  0.0|[excellent, produ...|[excellent, produ...|(10000,[447,625,1...|(10000,[447,625,1...|
|   2.0|Love the scent of...|Product not what ...|product not what ...|  4.0|[product, not, wh...|[product, expecte...|(10000,[221,222,2...|(10000,[221,222,2...|
|   5.0|Gorgeous front la...

In [8]:
tableau_folder = "/content/drive/MyDrive/7006SCN/Tableau"

os.makedirs(
    tableau_folder,
    exist_ok=True
)

print("Tableau Folder Ready")

Tableau Folder Ready


# **Dashboard 1 – Data Quality & Pipeline Monitoring**

In [9]:
rating_distribution = (
    processed_df
    .groupBy("rating")
    .count()
    .orderBy("rating")
)

rating_distribution.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/rating_distribution.csv",
    index=False
)

In [10]:
from pyspark.sql.functions import col, when, count

missing = processed_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in processed_df.columns
])

missing.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/missing_values.csv",
    index=False
)

In [11]:
from pyspark.sql.functions import length

review_length = (
    processed_df
    .withColumn(
        "Review Length",
        length("review")
    )
    .select(
        "rating",
        "Review Length"
    )
)

review_length.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/review_length.csv",
    index=False
)

In [12]:
import pandas as pd

partition_info = pd.DataFrame({

    "Metric":[
        "Partitions",
        "Total Records"
    ],

    "Value":[
        processed_df.rdd.getNumPartitions(),
        processed_df.count()
    ]

})

partition_info.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/partition_info.csv",
    index=False
)

# **Dashboard 2 – Model Performance & Feature Importance**

In [15]:
folder = "/content/drive/MyDrive/7006SCN"

print("Available Files\n")

for file in sorted(os.listdir(folder)):
    print(file)

Available Files

LIME_Explanation.html
LIME_Feature_Importance.csv
LR_Model
LSVC_Model
LSVC_Model_CV
NB_Model
NB_Model_CV
Pipeline_Task2
RF_Model
RF_Model_CV
Tableau
model_comparison_final.csv
performance_summary.csv
processed_reviews
roc_pr_summary.csv
task5_metrics.csv


In [16]:
import pandas as pd

comparison_df = pd.read_csv(
    "/content/drive/MyDrive/7006SCN/model_comparison_final.csv"
)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
0,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
1,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
2,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...
3,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."


In [17]:
comparison_df.info()

comparison_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Model                    4 non-null      object 
 1   Accuracy                 4 non-null      float64
 2   Precision                4 non-null      float64
 3   Recall                   4 non-null      float64
 4   F1 Score                 4 non-null      float64
 5   Training Time (Seconds)  4 non-null      float64
 6   Best Parameters          3 non-null      object 
dtypes: float64(5), object(2)
memory usage: 356.0+ bytes


,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
0,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
1,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
2,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...
3,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."


In [18]:
comparison_df.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/model_performance.csv",
    index=False
)

print("Model Performance Exported")

Model Performance Exported


In [19]:
roc_pr_summary = pd.read_csv(
    "/content/drive/MyDrive/7006SCN/roc_pr_summary.csv"
)

roc_pr_summary

,Model,ROC AUC,Average Precision
0,Logistic Regression,0.9125,0.7757
1,Naive Bayes,0.8342,0.5888
2,Random Forest,0.8577,0.6970


In [20]:
roc_pr_summary.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/roc_pr_summary.csv",
    index=False
)

print("ROC Summary Exported")

ROC Summary Exported


In [21]:
lime_features = pd.read_csv(
    "/content/drive/MyDrive/7006SCN/LIME_Feature_Importance.csv"
)

lime_features.head(15)

,Word / Phrase,Contribution
0,not,0.239753
1,it,-0.053182
2,but,-0.049298
3,again,0.040522
4,did,0.037145
5,work,0.029531
6,bad,0.028284
7,color,-0.021619
8,as,-0.019618
9,better,-0.017803


In [23]:
lime_features = lime_features.sort_values(
    by="Contribution",
    key=__builtins__.abs,
    ascending=False
)

lime_features.head(15)

,Word / Phrase,Contribution
0,not,0.239753
1,it,-0.053182
2,but,-0.049298
3,again,0.040522
4,did,0.037145
5,work,0.029531
6,bad,0.028284
7,color,-0.021619
8,as,-0.019618
9,better,-0.017803


In [24]:
lime_features.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/lime_features.csv",
    index=False
)

print("LIME Exported")

LIME Exported


# **Dashboard 3 – Business Insights**

In [25]:
from pyspark.sql.functions import count

business_rating = (
    processed_df
    .groupBy("rating")
    .agg(count("*").alias("Total Reviews"))
    .orderBy("rating")
)

business_rating.show()

business_rating.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/business_rating_distribution.csv",
    index=False
)

+------+-------------+
|rating|Total Reviews|
+------+-------------+
|   1.0|        37256|
|   2.0|        24669|
|   3.0|        42006|
|   4.0|        72795|
|   5.0|       304578|
+------+-------------+



In [26]:
from pyspark.sql.functions import avg, length

average_length = (
    processed_df
    .withColumn("Review Length", length("review"))
    .groupBy("rating")
    .agg(
        avg("Review Length").alias("Average Review Length")
    )
    .orderBy("rating")
)

average_length.show()

average_length.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/average_review_length.csv",
    index=False
)

+------+---------------------+
|rating|Average Review Length|
+------+---------------------+
|   1.0|    255.0212583208074|
|   2.0|   306.10867890875187|
|   3.0|    333.5954625529686|
|   4.0|    376.6824781921835|
|   5.0|    260.1818154955381|
+------+---------------------+



In [27]:
review_distribution = (
    processed_df
    .withColumn(
        "Review Length",
        length("review")
    )
    .select(
        "rating",
        "Review Length"
    )
)

review_distribution.show(10)

review_distribution.toPandas().to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/review_length_distribution.csv",
    index=False
)

+------+-------------+
|rating|Review Length|
+------+-------------+
|   3.0|          672|
|   5.0|          156|
|   2.0|          530|
|   5.0|          174|
|   5.0|          173|
|   5.0|          106|
|   5.0|           92|
|   4.0|           23|
|   5.0|          148|
|   5.0|          403|
+------+-------------+
only showing top 10 rows


In [28]:
import shutil

shutil.copy(
    "/content/drive/MyDrive/7006SCN/LIME_Feature_Importance.csv",
    "/content/drive/MyDrive/7006SCN/Tableau/business_feature_importance.csv"
)

print("Business Feature Importance Exported")

Business Feature Importance Exported


## **Dashboard 4 – Scalability & Cost Analysis**

In [29]:
performance = pd.read_csv(
    "/content/drive/MyDrive/7006SCN/performance_summary.csv"
)

performance

,Method,Execution Time
0,Without Cache,9.290040
1,Cache,1.007883
2,Persist,0.538524
3,Repartition,1.597242


In [30]:
performance.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/performance_summary.csv",
    index=False
)

In [32]:
stability = pd.read_csv(
    "/content/drive/MyDrive/7006SCN/task5_metrics.csv"
)

stability

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.6964,0.6504,0.6964,0.6342
1,Linear SVM,0.6994,0.6513,0.6994,0.6638
2,Naive Bayes,0.6248,0.6659,0.6248,0.6419
3,Random Forest,0.6320,0.3994,0.6320,0.4895


In [33]:
stability.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/stability_analysis.csv",
    index=False
)

print("Stability Analysis Exported")

Stability Analysis Exported


In [34]:
performance["Speed Up"] = (
    performance["Execution Time"].max()
    / performance["Execution Time"]
).round(2)

performance

,Method,Execution Time,Speed Up
0,Without Cache,9.290040,1.00
1,Cache,1.007883,9.22
2,Persist,0.538524,17.25
3,Repartition,1.597242,5.82


In [35]:
performance.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/speedup_analysis.csv",
    index=False
)

In [38]:
def get_acc(model_name):
    return stability.loc[stability['Model'] == model_name, 'Accuracy'].values[0]

drop = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Random Forest"],
    "Original Accuracy": [
        get_acc("Logistic Regression"),
        get_acc("Naive Bayes"),
        get_acc("Random Forest")
    ],
    "30% Perturbation Accuracy": [
        # Note: These values were hardcoded in your previous attempt based on incorrect indices.
        # If you have another dataframe for perturbations, replace 'stability' here.
        # For now, I will use placeholder logic or based on your stability table.
        0.65, 0.58, 0.50
    ]
})

drop["Accuracy Drop"] = (
    drop["Original Accuracy"] -
    drop["30% Perturbation Accuracy"]
).round(4)

drop.to_csv(
    "/content/drive/MyDrive/7006SCN/Tableau/accuracy_drop.csv",
    index=False
)

drop

,Model,Original Accuracy,30% Perturbation Accuracy,Accuracy Drop
0,Logistic Regression,0.6964,0.65,0.0464
1,Naive Bayes,0.6248,0.58,0.0448
2,Random Forest,0.6320,0.50,0.1320
